# Step 1 — 기저 준수율 smoke test

H1 하네스로 모델이 함수 10개를 순차 생성하게 하고, 함수명 표기 규칙 준수율을 잰다.

**목적** — 계획서 5.5 게이트(기저 준수율 55~85%) 통과 여부 확인 + 미결정 사항 두 개(모델 크기, 지침 방향) 결정.

**GPU 런타임이 필요하다.** 런타임 → 런타임 유형 변경 → T4 GPU 선택.

> Colab은 세션이 끊긴다. 결과는 `results/step1_baseline.jsonl` 에 세션 단위로 append되고,
> 재시작하면 완료된 `(model, cell, seed)` 조합을 건너뛴다. 끊기면 아래 실행 셀만 다시 돌리면 이어진다.

## 0. GPU 확인

In [ ]:
!nvidia-smi -L

## 1. repo clone

공개 저장소라 토큰 없이 clone된다. private으로 바꾸면 URL에 토큰을 넣어야 한다.

In [ ]:
REPO_URL = "https://github.com/deanjs/instruction-adherence.git"

import os

# 재시작 후에도 결과 JSONL을 보존하려고, 이미 clone돼 있으면 지우지 않는다.
if not os.path.exists("/content/repo"):
    !git clone -q $REPO_URL /content/repo
%cd /content/repo
!ls results/

## 2. 의존성 설치

Colab에는 torch가 사전 설치돼 있다(그 CUDA 빌드를 유지해야 하므로 재설치하지 않는다).
transformers/accelerate만 버전을 맞춘다. 설치 후 버전 경고가 뜨면 런타임을 한 번 재시작한다.

In [ ]:
!pip install -q "transformers>=4.51.0" "accelerate>=0.26.0"

import torch, transformers
print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())

## 3. 실행

1.5B → 3B 순으로 두 모델을 로드해 4개 셀 × 10세션을 돈다.

- 셀: `python_camel` / `python_snake` / `ts_snake` / `ts_camel`
- 셀당 10세션, 세션당 함수 10개 순차 생성 (앞 생성물이 context에 누적)
- temperature 0.7, top_p 0.95, seed는 세션마다 기록 (조건 간 공유)

세션 하나가 끝날 때마다 JSONL에 append된다. **런타임이 끊기면 이 셀만 다시 실행**하면
완료분을 건너뛰고 이어서 돈다. (전체는 T4에서 대략 수십 분~1시간대)

> 주의: `/content/repo` 는 Colab VM이 재활용되면 통째로 사라진다. 실험을 며칠에 걸쳐
> 이어서 돌릴 거면 아래 '3-b'처럼 Google Drive에 결과를 두거나, 세션 끝에 5장에서
> 결과를 내려받아 두어야 재개가 실제로 된다.

In [ ]:
!python src/step1_baseline.py

### 3-b. (선택) Google Drive에 결과를 두어 VM 재활용에도 재개하기

VM이 끊겨도 결과를 잃지 않으려면 Drive에 결과 파일을 두고 `--out` 으로 가리킨다.
위 3장 대신 이 셀을 쓰면 된다. (처음 실행 시 Drive 권한 승인 팝업이 뜬다.)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
OUT = "/content/drive/MyDrive/instruction-adherence/step1_baseline.jsonl"
os.makedirs(os.path.dirname(OUT), exist_ok=True)

!python src/step1_baseline.py --out "$OUT"

## 4. 게이트 판정

모델·셀별 전체 준수율과 위치 1 준수율(주 지표)을 본다. 계획서 5.5 게이트는 **55~85%**.

| 판정 | 의미 |
|---|---|
| PASS | 게이트 통과 — 이 (모델, 지침 방향)이 후보 |
| 천장 | >85% — 천장 효과, 낮출 여지 없음 (관용 표기가 이쪽일 가능성) |
| 바닥 | <55% — 신호가 노이즈에 묻힘 |

**결정 규칙**
- 모델 크기: 게이트를 통과하는 **가장 작은** 모델(1.5B 우선).
- 지침 방향: 언어별로 PASS가 나온 방향을 택한다. 관용 표기(python_snake / ts_camel)가
  천장이면 비관용 방향(python_camel / ts_snake)으로 뒤집는다.

> Drive를 썼다면 `--out` 을 똑같이 붙여 요약한다: `--summary-only --out "$OUT"`

In [ ]:
!python src/step1_baseline.py --summary-only

## 5. 결과를 내려받기 (선택)

`results/step1_baseline.jsonl` 은 사전 등록 기록이다. 로컬에서 커밋하려면 내려받는다.
(Drive를 썼다면 이미 Drive에 있으니 이 셀은 필요 없다.)

In [ ]:
from google.colab import files

files.download("results/step1_baseline.jsonl")